In [3]:
import pymupdf
import os
import sqlite3
import re

In [ ]:
conn = sqlite3.connect("slides.db")
cur = conn.cursor()

In [ ]:
cur.execute("SELECT DISTINCT filepath FROM slides")
already_ingested = set(row[0] for row in cur.fetchall())

In [48]:
def get_metadata(full_path, root):
    rel_path = os.path.relpath(full_path, root)
    parts = os.path.normpath(rel_path).split(os.sep)
    filename = parts[-1]
    if len(parts) == 5:
        year, semester, course, topic = parts[0], parts[1], parts[2], parts[3]
        
    elif len(parts) == 4:
        year, semester, course, topic = parts[0], parts[1], parts[2], None

    else:
        print(full_path)
        year, semester, course, topic = None, None, None, None
        
    return [year, semester, course, topic, filename]

In [50]:
def extract_pdf_text(pdf_path):
    doc = pymupdf.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()
        pages.append((page_num, text))
    doc.close()
    return pages     

In [ ]:
root = "C:/Users/obeid/OneDrive/Documents/Desktop/PSUT"
all_data = []
for dirpath, dirnames, filenames in os.walk(root):
    for filename in filenames:
        full_path = os.path.join(dirpath, filename)
        rel_path = os.path.relpath(full_path, root)
        if rel_path in already_ingested:
            continue
        if filename.endswith(".pdf") or filename.endswith(".pptx"):
            full_path = os.path.join(dirpath, filename)
            meta_data = get_metadata(full_path, root)
            pages = extract_pdf_text(full_path)
            
            for page_num, text in pages:
                text = re.sub(r'\s+', ' ', text).strip()
                if text== '':
                    continue
                row = meta_data + [page_num, text]
                all_data.append(row)
        

In [73]:
conn = sqlite3.connect("slides.db")
cur = conn.cursor()
cur.execute("""
            CREATE TABLE IF NOT EXISTS slides(
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                year TEXT,
                semester TEXT,
                course TEXT,
                topic TEXT,
                filename TEXT,
                pageNum INTEGER,
                text TEXT
            )
""")
conn.commit()

In [74]:
cur.executemany(
    "INSERT INTO slides (year, semester, course, topic, filename, pageNum, text) VALUES (?, ?, ?, ?, ?, ?, ?)",
    all_data
    )
conn.commit()

In [82]:
cur.execute("""
            CREATE VIRTUAL TABLE IF NOT EXISTS slides_fts5 USING FTS5(
                text,
                year,
                semester,
                course,
                topic,
                filename,
                pageNum
            )
            """
)
conn.commit()

In [83]:
cur.execute("INSERT INTO slides_fts5 (text, year, semester, course, topic, filename, pageNum) SELECT text, year, semester, course, topic, filename, pageNum FROM slides")
conn.commit()

In [7]:
cur.execute("select id, year, semester, course, topic, filename from slides")
rows = cur.fetchall()

for row_id, year, semester, course, topic, filename in rows:
    parts = [p for p in [year, semester, course, topic, filename] if p is not None]
    rel_path = os.path.join(*parts)
    cur.execute("update slides set filePath = ? where id = ?", (rel_path, row_id))
    
conn.commit()
    
    

In [9]:
cur.execute("SELECT DISTINCT filepath FROM slides")
already_ingested = set(row[0] for row in cur.fetchall())

In [11]:
conn

In [25]:
cur.execute("DELETE FROM slides WHERE filename = ?", ("Cooking-Basics.pdf",))
cur.execute("DELETE FROM slides_fts5 WHERE filename = ?", ("Cooking-Basics.pdf",))
conn.commit()

In [19]:
for dirpath, dirnames, filenames in os.walk(root):
    for filename in filenames:
        if filename.endswith(".pdf"):
            full_path = os.path.join(dirpath, filename)
            rel_path = os.path.relpath(full_path, root)
            if "hisham_obeidat" in filename:  # replace with part of the actual filename
                print("FOUND:", rel_path)

FOUND: Third Year\Second semester\Data Mining\hisham_obeidat.pdf


In [18]:
root = "C:/Users/obeid/OneDrive/Documents/Desktop/PSUT"


In [26]:
cur.execute("select * from slides WHERE filename = ?", ("Cooking-Basics.pdf",))
for i in cur.fetchall():
    print(i)

(5468, 'Third Year', 'Second semester', 'Data Mining', 'Cooking-Basics.pdf', 'Cooking-Basics.pdf', 1, 'Cooking Basics EAT SMART Get Your Family to the Table SP732', 'Third Year\\Second semester\\Data Mining\\Cooking-Basics.pdf')
(5469, 'Third Year', 'Second semester', 'Data Mining', 'Cooking-Basics.pdf', 'Cooking-Basics.pdf', 2, '1 Starting with the Basics: Food, Equipment and Knowledge Cooking Basics: Introduction This publication provides information you may find helpful when you prepare recipes. • Kitchen equipment (page 2) • Measuring (page 3) • Measurement equivalents (page 4) • Recipe preparation steps (page 5) • Cooking terms (pages 6-7) • Ingredient substitutions (pages 8-11) • Trimming the fat from our diets (pages 12-14) • Healthy cooking tips (page 15) • Cooking with herbs, spices and seasonings (pages 16-18) • Seasoning your food with less salt (page 19) • Food yields (pages 20-21) • Putting out a cooking fire (page 22) Use this information to prepare all your recipes. Then

In [27]:
cur.execute("DELETE FROM slides WHERE filename = ?", ("Cooking-Basics.pdf",))
print("slides deleted:", cur.rowcount)

cur.execute("DELETE FROM slides_fts5 WHERE filename = ?", ("Cooking-Basics.pdf",))
print("fts5 deleted:", cur.rowcount)

conn.commit()

slides deleted: 24
fts5 deleted: 24
